# Populate the Sub-400 Archive

Generate a Phase 3 seed pool using greedy danger-objective search with tabu memory. The batch uses both archived sub-600 and random starting colorings, and saves only distinct results with exact score 399 or lower.

In [1]:
# Imports and Project Root
from dataclasses import dataclass
from pathlib import Path
from time import perf_counter

import numpy as np

from ramsey import (
    RArchiveBatch,
    RArchiveBatchConfig,
    RArchiveSnapshotConstruction,
    RDangerObjective,
    REnvironment,
    REnvironmentConfig,
    RGraph,
    RGreedyPolicy,
    RProblem,
    RSearch,
    RSQLiteArchive,
    RTabuMemory,
    RTabuMemoryConfig,
)

project_root = Path.cwd().resolve()

if project_root.name == "notebooks":
    project_root = project_root.parent

if not (project_root / "ramsey").is_dir():
    raise RuntimeError(
        "Run this notebook from the RamseyNumber root "
        "or notebooks directory."
    )

In [2]:
# Batch Configuration

RANDOM_SEED = 202_608_052
N_VERTICES = 43

RUN_NAME = (
    "greedy-tabu-sub-400-"
    "phase-3-seeds-002"
)

TARGET_SUB_400_COLORINGS = 1_000
MAXIMUM_ATTEMPTS = 500
START_ITERATION = 0

TARGET_MAXIMUM_SCORE = 399
ARCHIVE_SEED_SCORE_LIMIT = 399
MINIMUM_ARCHIVE_SEEDS = 500

SEARCH_STEPS = 2_000
DANGER_DECAY = 0.25
EDGE_TABU_TENURE = 20
VISITED_STATE_WINDOW = 2_000
REPORT_INTERVAL = 10

DATABASE_PATH = (
    project_root
    / "data"
    / "ramsey_colorings.sqlite3"
)

In [3]:
# Runtime, Graph, and Existing Archive

rng = np.random.default_rng(
    RANDOM_SEED
)

graph = RGraph(
    RProblem.r55(
        n_vertices=N_VERTICES,
    )
)

existing_archive = globals().get("archive")

if existing_archive is not None:
    existing_archive.close()

archive = RSQLiteArchive(
    DATABASE_PATH
)

eligible_seed_count = (
    archive.coloring_count_in_score_range(
        maximum_score=ARCHIVE_SEED_SCORE_LIMIT,
        graph=graph,
    )
)

existing_target_count = (
    archive.coloring_count_in_score_range(
        maximum_score=TARGET_MAXIMUM_SCORE,
        graph=graph,
    )
)

if eligible_seed_count < MINIMUM_ARCHIVE_SEEDS:
    archive.close()

    raise RuntimeError(
        f"Expected at least {MINIMUM_ARCHIVE_SEEDS} "
        f"archived sub-{ARCHIVE_SEED_SCORE_LIMIT + 1} seeds; "
        f"found {eligible_seed_count}."
    )

print("Database:", DATABASE_PATH.resolve())
print("Archive best:", archive.best_score(graph))
print("Eligible archived seeds:", eligible_seed_count)
print("Existing sub-400 colorings:", existing_target_count)
print("Target sub-400 colorings:", TARGET_SUB_400_COLORINGS)

Database: C:\code\RamseyNumber\data\ramsey_colorings.sqlite3
Archive best: 286
Eligible archived seeds: 500
Existing sub-400 colorings: 500
Target sub-400 colorings: 1000


In [4]:
# Fixed Seed Snapshot

construction = (
    RArchiveSnapshotConstruction(
        archive=archive,
        rng=rng,
        maximum_score=(
            ARCHIVE_SEED_SCORE_LIMIT
        ),
    )
)

snapshot_size = construction.prepare(
    graph
)

print(
    "Construction:",
    construction.name,
)

print(
    "Frozen snapshot size:",
    snapshot_size,
)

print(
    "Seeds will be consumed "
    "without replacement."
)

Construction: archive-snapshot-score-any-to-399
Frozen snapshot size: 500
Seeds will be consumed without replacement.


In [5]:
# Greedy Tabu Search

objective = RDangerObjective(
    decay=DANGER_DECAY,
)

memory = RTabuMemory(
    number_of_edges=graph.number_of_edges,
    config=RTabuMemoryConfig(
        edge_tenure=EDGE_TABU_TENURE,
        visited_state_window=VISITED_STATE_WINDOW,
    ),
)

environment = REnvironment(
    graph=graph,
    objective=objective,
    memory=memory,
    config=REnvironmentConfig(
        max_steps=SEARCH_STEPS,
        use_aspiration=True,
    ),
)

policy = RGreedyPolicy(
    rng=rng,
    use_objective_reward=True,
)

search = RSearch(
    environment=environment,
    policy=policy,
)

print("Objective:", objective.name)
print("Policy:", policy.name)
print("Search steps per attempt:", SEARCH_STEPS)

Objective: danger
Policy: greedy-objective
Search steps per attempt: 2000


In [6]:
# Assemble the Strict Archive Batch

batch = RArchiveBatch(
    graph=graph,
    construction=construction,
    search=search,
    archive=archive,
)

batch_config = RArchiveBatchConfig(
    run_name=RUN_NAME,
    target_count=TARGET_SUB_400_COLORINGS,
    maximum_attempts=MAXIMUM_ATTEMPTS,
    maximum_score=TARGET_MAXIMUM_SCORE,
    start_iteration=START_ITERATION,
    record_steps=False,
    save_out_of_range=False,
)

print("Run name:", batch_config.run_name)
print("Maximum attempts:", batch_config.maximum_attempts)
print("Only qualifying results will be archived.")

Run name: greedy-tabu-sub-400-phase-3-seeds-002
Maximum attempts: 500
Only qualifying results will be archived.


In [7]:
# Progress Observer

progress = {
    "start": perf_counter(),
    "best": archive.best_score(graph),
}

def report_attempt(attempt_result):
    result = attempt_result.search_result

    new_database_best = (
        attempt_result.archive_record is not None
        and (
            progress["best"] is None
            or result.best_score < progress["best"]
        )
    )

    if new_database_best:
        progress["best"] = result.best_score

    should_report = (
        attempt_result.attempt % REPORT_INTERVAL == 0
        or attempt_result.in_score_range
        or new_database_best
    )

    if not should_report:
        return

    elapsed = perf_counter() - progress["start"]

    flags = []

    if attempt_result.in_score_range:
        flags.append("SUB-400")

    if attempt_result.new_unique_coloring:
        flags.append("NEW-UNIQUE")

    if new_database_best:
        flags.append("NEW-BEST")

    flag_text = (
        " | " + " | ".join(flags)
        if flags
        else ""
    )

    print(
        f"Attempt {attempt_result.attempt:5d} | "
        f"source={attempt_result.construction_name:24s} | "
        f"initial={result.initial_score:4d} | "
        f"final={result.final_score:4d} | "
        f"best={result.best_score:4d} | "
        f"eligible={attempt_result.eligible_count:4d}"
        f"/{TARGET_SUB_400_COLORINGS:4d} | "
        f"elapsed={elapsed:8.1f}s"
        f"{flag_text}"
    )

In [8]:
# Populate the Sub-400 Pool

batch_start = perf_counter()

batch_result = batch.populate(
    batch_config,
    observer=report_attempt,
)

batch_elapsed = perf_counter() - batch_start

print()
print("Attempts completed:", batch_result.attempts_completed)
print("Target reached:", batch_result.target_reached)
print("Initial eligible:", batch_result.initial_eligible_count)
print("Final eligible:", batch_result.final_eligible_count)
print("New eligible:", batch_result.new_eligible_colorings)
print("Best attempt score:", batch_result.best_score)
print("Elapsed:", f"{batch_elapsed:.3f} seconds")

if batch_result.attempts_completed:
    print(
        "Mean time per attempt:",
        f"{batch_elapsed / batch_result.attempts_completed:.3f} seconds",
    )

Attempt     0 | source=archive-snapshot-score-any-to-399 | initial= 323 | final= 350 | best= 318 | eligible= 501/1000 | elapsed=     6.4s | SUB-400 | NEW-UNIQUE
Attempt     1 | source=archive-snapshot-score-any-to-399 | initial= 328 | final= 369 | best= 313 | eligible= 502/1000 | elapsed=    11.9s | SUB-400 | NEW-UNIQUE
Attempt     2 | source=archive-snapshot-score-any-to-399 | initial= 333 | final= 321 | best= 290 | eligible= 503/1000 | elapsed=    17.8s | SUB-400 | NEW-UNIQUE
Attempt     3 | source=archive-snapshot-score-any-to-399 | initial= 333 | final= 371 | best= 298 | eligible= 504/1000 | elapsed=    23.1s | SUB-400 | NEW-UNIQUE
Attempt     4 | source=archive-snapshot-score-any-to-399 | initial= 339 | final= 327 | best= 295 | eligible= 505/1000 | elapsed=    29.2s | SUB-400 | NEW-UNIQUE
Attempt     5 | source=archive-snapshot-score-any-to-399 | initial= 323 | final= 374 | best= 308 | eligible= 506/1000 | elapsed=    34.9s | SUB-400 | NEW-UNIQUE
Attempt     6 | source=archive-sna

In [9]:
# Phase 3 Pool Summary

score_bands = (
    (0, 299),
    (300, 349),
    (350, 399),
)

print("Archive best:", archive.best_score(graph))

for minimum_score, maximum_score in score_bands:
    count = archive.coloring_count_in_score_range(
        minimum_score=minimum_score,
        maximum_score=maximum_score,
        graph=graph,
    )

    print(
        f"Scores {minimum_score:3d}–{maximum_score:3d}:",
        count,
    )

print(
    "Total sub-400:",
    archive.coloring_count_in_score_range(
        maximum_score=399,
        graph=graph,
    ),
)

Archive best: 278
Scores   0–299: 53
Scores 300–349: 693
Scores 350–399: 150
Total sub-400: 896


In [10]:
# Release the SQLite Connection

archive.close()
print("Archive closed.")

Archive closed.
